# 1 · Domain Shift messen: Leave-one-center-out

**Frage:** Wie viel Leistung verliert ein Modell, wenn es Bilder aus einem Spital sieht, das im Training nicht vorkam? Und wie viel holen Stain Normalization und Farbaugmentation davon zurück?

**Daten:** Camelyon17-WILDS – rund 450'000 Patches (96×96 px) aus Lymphknoten-Schnitten von 5 Spitälern, Label Tumor ja/nein.

**Aufbau**
1. Split nach Spital und Slide: Training auf Spitälern 0, 3, 4; Validierung auf *anderen Slides* derselben Spitäler (= ohne Domain Shift); Test auf Spital 1 und 2 (= mit Domain Shift).
2. Vier Varianten mit identischem Modell und Training: `baseline`, `macenko`, `hed` (Farbaugmentation), `macenko+hed`.
3. Ergebnis: Genauigkeit und AUC ohne und mit Domain Shift; die Differenz ist der Preis des Shifts.
4. Zeitmessungen für die Kostenfrage am Ende.

**Hardware:** GPU empfohlen (CUDA oder Apple MPS). Auf CPU `N_TRAIN_PER_HOSP` und `N_EVAL_PER_HOSP` stark reduzieren. Der Download beträgt rund 10 GB.

In [ ]:
# Einmalig installieren (oder: pip install -r requirements.txt)
# %pip install -r requirements.txt

import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import stainlib as sl
import trainlib as tl

DEVICE = tl.get_device()
print("Gerät:", DEVICE)

## Konfiguration

In [ ]:
DATA_ROOT = "data"
TRAIN_HOSP = [0, 3, 4]          # offizieller WILDS-Split; Spitäler 1 und 2 sind "fremd"
N_TRAIN_PER_HOSP = 10_000       # Trainingspatches pro Spital (ausgewogen Tumor/Normal)
N_VAL_SLIDES_PER_HOSP = 2       # zurückgehaltene Slides pro Trainingsspital für die ID-Validierung
N_EVAL_PER_HOSP = 4_000         # Evaluationspatches pro Spital
SEEDS = [0, 1, 2]               # drei Seeds für Mittelwert und Streuung

# ------------------------------------------------------------------------------------------
# LAUF wählen – nur diese eine Zeile ändern:
#   "standard" : vortrainiert + Drehung/Spiegelung, 3 Epochen, HED σ 0.1, Original-Macenko (bereits gelaufen)
#   "scratch"  : von Grund auf, ohne Drehung/Spiegelung, 5 Epochen (bereits gelaufen)
#   "dark"     : wie "standard", aber Macenko mit Filter für schwarze Pixel. Nur die Macenko-Varianten,
#                weil sich baseline und hed dadurch nicht ändern.
#   "tellez"   : von Grund auf, 10 Epochen, starke HED-Augmentation σ 0.2 (wie Tellez "HED-strong"),
#                Macenko mit Filter. Alle vier Varianten – prüft, ob Augmentation zu Macenko aufholt.
#   "dark60"   : wie "dark", aber strengerer Filter (hellster Kanal < 60) – erfasst auch sehr dunkle Pixel
#                (z. B. Spital 1). Nur die Macenko-Varianten. (bereits gelaufen)
#   "scratch60": wie "scratch", aber Macenko mit Filter < 60. Nur die Macenko-Varianten – baseline und hed
#                aus dem Lauf "scratch" bleiben gültig, weil der Filter sie nicht berührt.
#                Zusammen mit "dark60" ergibt das die korrigierten Zahlen für beide Bedingungen.
LAUF = "scratch60"
# ------------------------------------------------------------------------------------------
BASE = dict(arch="resnet18", batch_size=128, lr=3e-4, workers=4)
PRESETS = {
    "standard": dict(pretrained=True,  geo_aug=True,  epochs=3,  hed_sigma=0.1, macenko_dark=None),
    "scratch":  dict(pretrained=False, geo_aug=False, epochs=5,  hed_sigma=0.1, macenko_dark=None),
    "dark":     dict(pretrained=True,  geo_aug=True,  epochs=3,  hed_sigma=0.1, macenko_dark=30),
    "tellez":   dict(pretrained=False, geo_aug=False, epochs=10, hed_sigma=0.2, macenko_dark=30),
    "dark60":   dict(pretrained=True,  geo_aug=True,  epochs=3,  hed_sigma=0.1, macenko_dark=60),
    "scratch60": dict(pretrained=False, geo_aug=False, epochs=5, hed_sigma=0.1, macenko_dark=60),
}
VARIANTS = (["macenko", "macenko+hed"] if LAUF in ("dark", "dark60", "scratch60")
            else ["baseline", "macenko", "hed", "macenko+hed"])
CFG = {**BASE, **PRESETS[LAUF]}
# macenko_dark: Pixel, deren hellster Kanal < Wert ist (fast schwarz), fliessen nicht in die Macenko-Schätzung ein.

RUN = tl.run_suffix(CFG)        # "" für "standard", sonst z. B. "_dark" oder "_scratch_nogeo_ep10_hed0.2_dark"
STAIN_FILE = ("results/stain_params.pkl" if CFG["macenko_dark"] is None
              else "results/stain_params_dark.pkl" if CFG["macenko_dark"] == 30
              else f"results/stain_params_dark{CFG['macenko_dark']}.pkl")
Path("results").mkdir(exist_ok=True)
print("Lauf:", LAUF, "| Dateizusatz:", RUN or "(keiner)", "| Varianten:", VARIANTS, "| Epochen:", CFG["epochs"])


## Daten laden und ansehen

In [ ]:
wds, meta = tl.load_camelyon(DATA_ROOT, download=True)
overview = (pd.DataFrame(meta).groupby("hospital")
            .agg(patches=("y", "size"), tumor_anteil=("y", "mean"), slides=("slide", "nunique")))
overview

In [ ]:
split = tl.make_split(meta, TRAIN_HOSP, N_TRAIN_PER_HOSP, N_VAL_SLIDES_PER_HOSP, N_EVAL_PER_HOSP, seed=0)
tl.save_json(split, "results/split.json")
print("Training:", len(split["train"]), "| ID-Validierung:", len(split["id_val"]),
      "| OOD:", {h: len(v) for h, v in split["ood"].items()})

## Macenko-Parameter: einmal pro Slide

Stain-Matrix und Intensität werden pro Slide aus 32 zufälligen Patches geschätzt und dann auf alle Patches dieses Slides angewandt. Das vermeidet Sprünge zwischen Kacheln und Fehlschätzungen bei Patches mit wenig Kernen. Die Schätzung nutzt keine Labels und ist damit auch auf Testslides zulässig.

Die **Referenz** wird aus einem Mosaik von 256 Trainingspatches geschätzt.

**Filter für schwarze Pixel** (`macenko_dark`): Schwarze und sehr dunkle Pixel (Ursache nicht geprüft – denkbar sind Scan-Ränder, Tinte oder sehr dichte Kerne) liegen im OD-Raum auf der Graudiagonale und ziehen den Hämatoxylin-Vektor dorthin. Erkennbar an einer H-Intensität von genau √3·ln 240 ≈ 9,49. Die Zelle unten zählt diese Slides.


In [ ]:
t0 = time.perf_counter()
ref = tl.fit_reference(wds, split["train"], k=256, dark_thresh=CFG["macenko_dark"])
t_ref = time.perf_counter() - t0

t0 = time.perf_counter()
slide_params = tl.fit_slide_params(wds, meta, k=32, dark_thresh=CFG["macenko_dark"])
t_fit = time.perf_counter() - t0
n_slides = len(np.unique(meta["slide"]))
print(f"Referenz: {t_ref:.1f} s | Slide-Fit: {t_fit:.1f} s für {n_slides} Slides "
      f"({t_fit / n_slides:.2f} s pro Slide)")

# Kontrolle: Slides, deren H-Vektor von schwarzen Pixeln verdorben ist (H-Intensität ≈ 9,49, Winkel ≈ 0°)
def suspicious(params):   # H-Vektor fast auf der Graudiagonale (< 5°) oder H-Intensität eines schwarzen Pixels
    return [s for s, p in params.items() if p.maxC[0] > 9.0 or p.angles_deg()[0] < 5.0]
bad = suspicious(slide_params)
print(f"Verdächtige Slides (H-Winkel < 5° oder H-Intensität > 9): {len(bad)} von {len(slide_params)} {bad}")
if CFG["macenko_dark"] is not None:   # zum Vergleich: dieselbe Schätzung ohne Filter (dauert nur Sekunden)
    bad0 = suspicious(tl.fit_slide_params(wds, meta, k=32, verbose=False))
    print(f"Ohne Filter wären es: {len(bad0)} von {len(slide_params)} {bad0}")

tl.save_pickle({"ref": ref, "slide_params": slide_params}, STAIN_FILE)
print("Gespeichert:", STAIN_FILE)


In [ ]:
# Beispiele: je Spital zwei Patches, oben roh, unten Macenko-normalisiert
rng = np.random.default_rng(1)
fig, ax = plt.subplots(2, 10, figsize=(15, 3.4))
for h in range(5):
    idx = rng.choice(np.where(meta["hospital"] == h)[0], 2, replace=False)
    for j, i in enumerate(idx):
        img = tl.load_image(wds, i)
        p = slide_params.get(int(meta["slide"][i]))
        ax[0, 2 * h + j].imshow(img)
        ax[1, 2 * h + j].imshow(sl.macenko_apply(img, p, ref) if p else img)
        ax[0, 2 * h + j].set_title(f"Spital {h}", fontsize=9)
for a in ax.ravel():
    a.axis("off")
ax[0, 0].text(-20, 48, "roh", rotation=90, va="center")
ax[1, 0].text(-20, 48, "Macenko", rotation=90, va="center")
plt.tight_layout(); plt.show()

## Bilder für die Folien exportieren

In [ ]:
# Pro Spital ein 3x3-Mosaik aus den gewebereichsten Patches eines Slides, roh und Macenko-normalisiert.
# Ergebnis: results/slides/hosp{h}_raw.png und hosp{h}_macenko.png (je 1152 x 1152 px).
from PIL import Image

OUT = Path("results/slides"); OUT.mkdir(parents=True, exist_ok=True)
SCALE = 4   # 96 px -> 384 px; NEAREST erfindet keine Details

def tile(imgs, cols=3):
    return np.concatenate([np.concatenate(imgs[i:i + cols], axis=1)
                           for i in range(0, len(imgs), cols)], axis=0)

def save(arr, name):
    im = Image.fromarray(arr)
    im.resize((im.width * SCALE, im.height * SCALE), Image.NEAREST).save(OUT / name)

rng = np.random.default_rng(7)
for h in range(5):
    s = int(rng.choice(np.unique(meta["slide"][meta["hospital"] == h])))
    idx = np.where(meta["slide"] == s)[0]
    cand = [tl.load_image(wds, i) for i in rng.choice(idx, size=min(200, len(idx)), replace=False)]
    top = [cand[k] for k in np.argsort([sl.tissue_mask(im).mean() for im in cand])[::-1][:9]]
    p = slide_params.get(s)
    save(tile(top), f"hosp{h}_raw.png")
    save(tile([sl.macenko_apply(im, p, ref) if p else im for im in top]), f"hosp{h}_macenko.png")
print(sorted(f.name for f in OUT.iterdir()))

## Vorverarbeitungskosten pro Patch (ein CPU-Kern)

In [ ]:
bench = []
_ = tl.benchmark_transform(tl.PatchDataset(wds, split["train"][:500], meta), 500)  # Aufwärmen: Dateien in den Cache
for v in dict.fromkeys(["baseline", "macenko"] + VARIANTS):   # baseline und macenko immer, als Bezug für die Kosten
    ds = tl.PatchDataset(wds, split["train"][:500], meta, mode=v, slide_params=slide_params, ref=ref,
                         train=True, hed_sigma=CFG["hed_sigma"])
    bench.append({"variant": v, "ms_pro_patch": tl.benchmark_transform(ds, 500)})
bench = pd.DataFrame(bench)
bench.to_csv(f"results/transform_benchmark{RUN}.csv", index=False)
bench

## Training und Evaluation

Gleiches Modell (ResNet-18, je nach Lauf ImageNet-vortrainiert oder von Grund auf), gleiche Daten, gleiche Hyperparameter – nur die Farbbehandlung unterscheidet sich.

In [ ]:
rows, timings = tl.run_experiment(wds, meta, split, VARIANTS, SEEDS, slide_params, ref, CFG)
res = pd.DataFrame(rows)
res.to_csv(f"results/nb1_results{RUN}.csv", index=False)
tl.save_json(timings, f"results/nb1_timings{RUN}.json")
res

## Ergebnis: Preis des Domain Shifts und was die Varianten zurückholen

In [ ]:
res["setting"] = np.where(res["eval"] == "id_val", "ohne Shift (ID)", "mit Shift (OOD)")
# OOD = Mittel über die fremden Spitäler, dann über Seeds
per_seed = res.groupby(["variant", "seed", "setting"])[["acc", "auc"]].mean().reset_index()
summary = per_seed.groupby(["variant", "setting"])[["acc", "auc"]].agg(["mean", "std"]).round(3)
gap = (per_seed.pivot_table(index=["variant", "seed"], columns="setting", values="acc")
       .assign(gap=lambda d: d["ohne Shift (ID)"] - d["mit Shift (OOD)"])
       .groupby("variant")["gap"].agg(["mean", "std"]).round(3))
display(summary)
print("Genauigkeitsverlust durch den Domain Shift (ID − OOD):")
display(gap)

In [ ]:
order = [v for v in VARIANTS if v in per_seed["variant"].unique()]
m = per_seed.groupby(["variant", "setting"])["acc"].mean().unstack().loc[order]
s = per_seed.groupby(["variant", "setting"])["acc"].std().unstack().loc[order].fillna(0)
ax = m[["ohne Shift (ID)", "mit Shift (OOD)"]].plot.bar(yerr=s[["ohne Shift (ID)", "mit Shift (OOD)"]],
                                                         figsize=(8, 4), rot=0, capsize=3)
ax.set_ylabel("Genauigkeit"); ax.set_xlabel(""); ax.set_ylim(0.5, 1.0)
ax.set_title("Camelyon17-WILDS: Leistung ohne und mit Domain Shift")
for c in ax.containers:
    if hasattr(c, "datavalues"):
        ax.bar_label(c, fmt="%.2f", fontsize=8, padding=2)
plt.tight_layout(); plt.savefig(f"results/nb1_id_vs_ood{RUN}.png", dpi=200); plt.show()

# Pro fremdem Spital
display(res[res["eval"] != "id_val"].pivot_table(index="variant", columns="eval", values="acc").loc[order].round(3))

**So liest du das Ergebnis**
- Der Abstand zwischen den Balken von `baseline` ist der Preis des Domain Shifts. Die Literatur berichtet für diesen Datensatz rund 93 % → 70 % (Koh et al. 2021, mit anderem Modell und Training); deine Zahlen werden davon abweichen.
- Wie viel `macenko`, `hed` und `macenko+hed` von diesem Abstand zurückholen, ist die eigentliche Antwort auf „Lohnt sich Normalisierung?“.
- Mit nur einem Seed schwanken die Werte spürbar. Für belastbare Aussagen `SEEDS = [0, 1, 2]` und Fehlerbalken zeigen.

## Bedingungen vergleichen

Liest alle Ergebnisdateien `results/nb1_results*.csv` ein, z. B. die Standardbedingung und die WILDS-artige (`_scratch_nogeo`). Sinnvoll, sobald mindestens zwei Bedingungen gelaufen sind.

In [ ]:
import glob
allres = []
for f in sorted(glob.glob("results/nb1_results*.csv")):
    d = pd.read_csv(f)
    d["bedingung"] = Path(f).stem.replace("nb1_results", "") or "_standard"
    allres.append(d)
allres = pd.concat(allres, ignore_index=True)
allres["setting"] = np.where(allres["eval"] == "id_val", "ID", "OOD")
ps = allres.groupby(["bedingung", "variant", "seed", "setting"])["acc"].mean().reset_index()
tab = ps.pivot_table(index=["bedingung", "variant"], columns="setting", values="acc", aggfunc=["mean", "std"]).round(3)
tab[("mean", "Lücke ID−OOD")] = (tab[("mean", "ID")] - tab[("mean", "OOD")]).round(3)
display(tab)
allres.to_csv("results/nb1_all_conditions.csv", index=False)

## Kosten: Rechenaufwand aus den Messungen hochrechnen

Die folgenden Annahmen sind **Platzhalter** – bitte durch eigene Werte ersetzen. Gerechnet werden nur Rechenkosten; Datenerhebung, Annotation und Validierung sind meist deutlich teurer (siehe Begleitdokument).

In [ ]:
# --- Annahmen (anpassen) ---
GPU_PREIS_CHF_H = 2.0        # Cloud-GPU pro Stunde
CPU_PREIS_CHF_H = 0.05       # pro CPU-Kern und Stunde
PATCHES_PRO_WSI = 20_000     # Gewebekacheln pro Whole-Slide-Image in deiner Auflösung
WSI_PRO_JAHR = 10_000        # im Einsatz verarbeitete Slides pro Jahr
TRAININGS_PRO_JAHR = 20      # Trainingsläufe inkl. Experimente
SKALIERUNG_TRAINING = 10     # echtes Training ist z. B. 10x grösser als dieses Experiment

t = pd.DataFrame([{"variant": k.split("|")[0], **{kk: vv for kk, vv in v.items() if kk != "epoch_s"}}
                  for k, v in timings.items()]).groupby("variant").mean(numeric_only=True)
t = t.join(bench.set_index("variant"))
t["train_gpu_h"] = t["train_s"] / 3600 * SKALIERUNG_TRAINING
t["train_chf_pro_jahr"] = t["train_gpu_h"] * GPU_PREIS_CHF_H * TRAININGS_PRO_JAHR

# Laufzeit-Mehraufwand der Normalisierung beim Einsatz: Anwenden pro Patch + Fit pro Slide
apply_ms = (bench.set_index("variant").loc["macenko", "ms_pro_patch"]
            - bench.set_index("variant").loc["baseline", "ms_pro_patch"])
fit_s_pro_wsi = t_fit / n_slides
norm_cpu_h_pro_jahr = (apply_ms / 1000 * PATCHES_PRO_WSI + fit_s_pro_wsi) * WSI_PRO_JAHR / 3600
print(f"Normalisierung im Einsatz: {apply_ms:.2f} ms/Patch + {fit_s_pro_wsi:.1f} s/Slide "
      f"→ {norm_cpu_h_pro_jahr:,.0f} CPU-Kern-Stunden/Jahr ≈ CHF {norm_cpu_h_pro_jahr * CPU_PREIS_CHF_H:,.0f}")
display(t[["train_s", "images_per_s", "data_wait_s", "ms_pro_patch", "train_gpu_h", "train_chf_pro_jahr"]].round(2))

## Optional: vollständiges Leave-one-center-out

Jedes Spital wird einmal zurückgehalten, trainiert wird auf den vier übrigen. Das ergibt fünf Datenpunkte pro Variante und erlaubt in Notebook 3, die gemessene Verschiebung eines Spitals mit seinem Leistungsverlust zu vergleichen. Laufzeit: etwa fünfmal die des Hauptexperiments.

In [ ]:
RUN_FULL_LOCO = False
LOCO_VARIANTS = ["baseline", "macenko", "hed"]

if RUN_FULL_LOCO:
    loco_rows = []
    for held_out in range(5):
        train_h = [h for h in range(5) if h != held_out]
        sp = tl.make_split(meta, train_h, N_TRAIN_PER_HOSP * 3 // 4, N_VAL_SLIDES_PER_HOSP, N_EVAL_PER_HOSP, seed=0)
        r, _ = tl.run_experiment(wds, meta, sp, LOCO_VARIANTS, [0], slide_params, ref, CFG, model_dir=None)
        loco_rows += [dict(row, held_out=held_out) for row in r]
    loco = pd.DataFrame(loco_rows)
    loco.to_csv("results/nb1_loco.csv", index=False)
    display(loco.pivot_table(index="held_out", columns=["variant", "eval"], values="acc").round(3))